In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*.csv"):
    print(p)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(
    "/kaggle/input/competitions/"
    "shl-hiring-assessment-2026/Dataset_Final"
)

FEATURE_PATH = Path(
    "/kaggle/input/datasets/"
    "saicharanbhimavarapu/"
    "shl-grammar-v2-features-copy/"
    "engineered_features.csv"
)

train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
features = pd.read_csv(FEATURE_PATH)

assert len(train) == 769
assert len(test) == 216
assert features.shape == (985, 55)
assert features.select_dtypes(include="number").shape[1] == 55
assert np.isfinite(features.to_numpy()).all()

X_train = features.iloc[:769].reset_index(drop=True)
X_test = features.iloc[769:].reset_index(drop=True)
y_train = train["label"].astype(float).reset_index(drop=True)

print("===== SHL V2 DATA PREPARATION =====")
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training labels:", y_train.shape)
print("Label range:", y_train.min(), "to", y_train.max())
print("Missing values:", X_train.isna().sum().sum())
print("Data preparation completed.")

In [ ]:
%pip -q install catboost

In [ ]:
import catboost

print("CatBoost version:", catboost.__version__)
print("CatBoost imported successfully!")

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X_train))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):

    model = CatBoostRegressor(
        iterations=800,
        depth=6,
        learning_rate=0.04,
        loss_function="RMSE",
        l2_leaf_reg=3,
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    model.fit(
        X_train.iloc[tr_idx],
        y_train.iloc[tr_idx]
    )

    predictions = model.predict(X_train.iloc[val_idx])
    oof[val_idx] = predictions

    fold_rmse = np.sqrt(
        mean_squared_error(y_train.iloc[val_idx], predictions)
    )
    fold_scores.append(fold_rmse)

    print(f"Fold {fold}: RMSE = {fold_rmse:.6f}")

rmse = np.sqrt(mean_squared_error(y_train, oof))
mae = mean_absolute_error(y_train, oof)
pearson = pearsonr(y_train, oof).statistic

print("\n===== V2 BASELINE DIAGNOSTIC =====")
print(f"OOF RMSE: {rmse:.6f}")
print(f"OOF MAE: {mae:.6f}")
print(f"OOF Pearson: {pearson:.6f}")

print("\nV1 reference RMSE: 0.704620")
print("V1 reference Pearson: 0.822992")

print("\nRMSE difference:", round(rmse - 0.704620, 6))

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import numpy as np
import pandas as pd

configs = [
    {"depth": 4, "learning_rate": 0.035,
     "iterations": 1000, "l2_leaf_reg": 5},

    {"depth": 5, "learning_rate": 0.035,
     "iterations": 1000, "l2_leaf_reg": 5},

    {"depth": 6, "learning_rate": 0.03,
     "iterations": 1200, "l2_leaf_reg": 7},

    {"depth": 5, "learning_rate": 0.025,
     "iterations": 1400, "l2_leaf_reg": 10},

    {"depth": 4, "learning_rate": 0.05,
     "iterations": 800, "l2_leaf_reg": 8},
]

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for i, params in enumerate(configs, 1):

    print(f"\nTraining configuration {i}/{len(configs)}")

    predictions = np.zeros(len(X_train))

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(X_train), 1
    ):
        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            thread_count=-1
        )

        model.fit(
            X_train.iloc[train_idx],
            y_train.iloc[train_idx]
        )

        predictions[val_idx] = model.predict(
            X_train.iloc[val_idx]
        )

    rmse = np.sqrt(
        mean_squared_error(y_train, predictions)
    )
    mae = mean_absolute_error(y_train, predictions)
    corr = pearsonr(y_train, predictions).statistic

    results.append({
        "config": i,
        "RMSE": rmse,
        "MAE": mae,
        "Pearson": corr,
        **params
    })

    print(
        f"RMSE: {rmse:.6f} | "
        f"Pearson: {corr:.6f}"
    )

results_df = pd.DataFrame(results).sort_values("RMSE")

print("\n===== CATBOOST V2 OPTIMIZATION RESULTS =====")
display(results_df)

best = results_df.iloc[0]
print("\nBest configuration:", int(best["config"]))
print("Best RMSE:", round(best["RMSE"], 6))
print("Best Pearson:", round(best["Pearson"], 6))

results_df.to_csv(
    "/kaggle/working/v2_tuning_results.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# Best model from V2 optimization
best_params = {
    "iterations": 1200,
    "depth": 6,
    "learning_rate": 0.03,
    "l2_leaf_reg": 7,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False
}

model_v2 = CatBoostRegressor(**best_params)

# Train on all labeled recordings
model_v2.fit(X_train, y_train)

# Training predictions
train_pred = model_v2.predict(X_train)

train_rmse = np.sqrt(
    mean_squared_error(y_train, train_pred)
)

train_mae = mean_absolute_error(
    y_train, train_pred
)

train_pearson = pearsonr(
    y_train, train_pred
).statistic

print("===== SHL CATBOOST V2 RESULTS =====")
print(f"Training RMSE: {train_rmse:.6f}")
print(f"Training MAE: {train_mae:.6f}")
print(f"Training Pearson: {train_pearson:.6f}")

print("\nValidation RMSE: 0.694009")
print("Validation Pearson: 0.829263")
print("Validation MAE: 0.542813")

# Generate predictions for test recordings
test_pred = model_v2.predict(X_test)
test_pred = np.clip(test_pred, 0, 5)

submission_v2 = pd.DataFrame({
    "filename": test["filename"],
    "label": test_pred
})

# Submission integrity checks
assert len(submission_v2) == 216
assert submission_v2["filename"].tolist() == test["filename"].tolist()
assert submission_v2["filename"].is_unique
assert submission_v2["label"].notna().all()
assert submission_v2["label"].between(0, 5).all()

submission_v2.to_csv(
    "/kaggle/working/submission_v2.csv",
    index=False
)

model_v2.save_model(
    "/kaggle/working/catboost_v2.cbm"
)

print("\nV2 submission saved successfully!")
print("File: /kaggle/working/submission_v2.csv")
print("Rows:", len(submission_v2))

display(submission_v2.head(10))

In [ ]:
import numpy as np
import pandas as pd

# V2 predictions already generated
v2 = submission_v2.copy()

print("===== V2 PREDICTION VALIDATION =====")

print("\nPrediction statistics:")
print(v2["label"].describe())

print("\nChecks:")
print("Total predictions:", len(v2))
print("Missing values:", v2["label"].isna().sum())
print("Duplicate filenames:", v2["filename"].duplicated().sum())
print("All scores within 0-5:", v2["label"].between(0, 5).all())

assert len(v2) == 216
assert v2["filename"].is_unique
assert v2["label"].notna().all()
assert np.isfinite(v2["label"]).all()
assert v2["label"].between(0, 5).all()
assert v2["filename"].tolist() == test["filename"].tolist()

print("\nV2 submission checks passed!")